# Experiment 10 — Boundary-Specific Representation Learning for CORN — Colab Run

task_3's MiT-B1+CORN pools the encoder's final feature map into one 512-dim vector *before* `CornHead`'s single `Linear(512,3)` decides all three ordinal boundaries from it -- the three decisions only ever differ in their linear weight row, never in what representation they read. This experiment gives each boundary its own learned spatial attention over the (un-pooled) feature map, added as a residual on top of the **frozen, already-trained RGB MiT-B1+CORN baseline** (same per-fold checkpoint exp5/6/7 load) -- Phase A convention: backbone and the baseline's own `CornHead` frozen throughout, only the new per-boundary heads train.

Three modes share one architecture (`scripts/model.py`'s `BoundaryCornHead`):
- `heads` -- boundary-specific nonlinear decision head, no spatial attention (every boundary reads the same pooled GAP vector).
- `attention` -- adds the learned per-boundary spatial gate `A_k = sigmoid(Conv1x1_k(F))`.
- `contrastive` -- same as `attention`, plus a per-boundary supervised contrastive term pulling same-side images together and pushing opposite-side images apart in that boundary's own representation space.

No new data, mask, or derived modality -- same original PIID staging photos, same 3-fold split as task_3's own baseline. See `README.md` for the full architecture writeup and the critical caveat this run is specifically meant to test: three independent attention modules can converge to near-identical spatial focus, making the design a no-op reparameterization regardless of what accuracy says -- `scripts/diagnose_attention.py` (run at the end of this notebook) measures that directly.

## 1. Mount Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Clone / update the repo

In [7]:
import os

REPO_DIR = "/content/CanRollasThesisProject"
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/canrollas/CanRollasThesisProject.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

remote: Enumerating objects: 13, done.
remote: Counting objects: 100% (13/13), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 7 (delta 6), reused 7 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (7/7), 862 bytes | 431.00 KiB/s, done.
From https://github.com/canrollas/CanRollasThesisProject
   6c7e1dc..32058fe  main       -> origin/main
Updating 6c7e1dc..32058fe
Fast-forward
 .../exp/exp10_boundary_specific_representation/scripts/test_model.py    | 2 ++
 1 file changed, 2 insertions(+)


## 3. Fetch the PIID staging dataset

Same `wound-stage-classification.zip` task_3's own notebook uses -- the ORIGINAL, unmodified RGB photos. No tissue, depth, or ROI dataset is needed for this experiment.

In [3]:
import glob

ZIP_PATH = "/content/drive/MyDrive/thesis/wound-stage-classification.zip"
EXTRACT_DIR = "/content/data"
DATASET_NAME = "wound-stage-classification"
DATASET_DIR = f"{REPO_DIR}/datasets/{DATASET_NAME}"

if not glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True):
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    !unzip -q "{ZIP_PATH}" -d {EXTRACT_DIR}

candidates = glob.glob(f"{EXTRACT_DIR}/**/{DATASET_NAME}", recursive=True)
assert candidates, f"couldn't find an extracted '{DATASET_NAME}' folder under {EXTRACT_DIR}"
os.makedirs(os.path.dirname(DATASET_DIR), exist_ok=True)
if not os.path.exists(DATASET_DIR):
    os.symlink(candidates[0], DATASET_DIR)
print("dataset at:", DATASET_DIR, "->", os.path.realpath(DATASET_DIR))

dataset at: /content/CanRollasThesisProject/datasets/wound-stage-classification -> /content/data/wound-stage-classification


## 4. Install dependencies

In [4]:
!pip install -q -r {REPO_DIR}/task_3/requirements.txt

## 5. RGB baseline checkpoints (fold0/1/2)

First tries to copy them from `drive/MyDrive/thesis/task_3_outputs/checkpoints/` (task_3's own Colab notebook output location). Only whichever folds are still missing after that get trained -- task_3's OWN, unmodified ablation script, restricted to `mit_b1__corn` (not a different/new RGB model for this experiment).

In [5]:
import glob
import os
import shutil

os.chdir(f"{REPO_DIR}/task_3")
CKPT_DIR = "results/task_3_outputs/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)

DRIVE_CKPT_DIR = "/content/drive/MyDrive/thesis/task_3_outputs"
all_drive_pt_files = glob.glob(f"{DRIVE_CKPT_DIR}/**/*.pt", recursive=True)
print(f"{len(all_drive_pt_files)} .pt file(s) found under {DRIVE_CKPT_DIR}")

for fold in range(3):
    local_path = f"{CKPT_DIR}/mit_b1__corn_fold{fold}.pt"
    if os.path.exists(local_path):
        continue
    candidates = [
        p for p in all_drive_pt_files
        if "mit_b1" in p.lower() and "corn" in p.lower() and f"fold{fold}" in p.lower()
    ]
    if candidates:
        shutil.copy(candidates[0], local_path)
        print(f"fold{fold}: copied from Drive ({candidates[0]})")
    else:
        print(f"fold{fold}: NOT found on Drive, will train below")

missing = [f for f in range(3) if not os.path.exists(f"{CKPT_DIR}/mit_b1__corn_fold{f}.pt")]
if missing:
    folds_arg = ",".join(str(f) for f in missing)
    !python run_ablation.py --configs mit_b1__corn --folds {folds_arg}
else:
    print("all 3 RGB baseline checkpoints present, nothing to train")

79 .pt file(s) found under /content/drive/MyDrive/thesis/task_3_outputs
fold0: copied from Drive (/content/drive/MyDrive/thesis/task_3_outputs/checkpoints/mit_b1__corn_fold0.pt)
fold1: copied from Drive (/content/drive/MyDrive/thesis/task_3_outputs/checkpoints/mit_b1__corn_fold1.pt)
fold2: copied from Drive (/content/drive/MyDrive/thesis/task_3_outputs/checkpoints/mit_b1__corn_fold2.pt)
all 3 RGB baseline checkpoints present, nothing to train


## 6. Architecture sanity check

No checkpoint needed -- frozen-baseline-sub-computation equality, attention-gradient-flow, and freeze/unfreeze toggle checks.

In [8]:
os.chdir(f"{REPO_DIR}/more_experiments/exp/exp10_boundary_specific_representation/scripts")
!python test_model.py

[PASS] final stage shape [B,512,7,7]
[PASS] FEATURE_DIM == 512
[PASS] 'heads' mode: z_k == plain GAP(F) for every boundary (no attention)
[PASS] 'heads' mode: 3 boundary logits produced
[PASS] [heads] pooled GAP matches reference backbone exactly
[PASS] [heads] frozen baseline_logits sub-computation matches reference CornHead exactly
[PASS] [heads] baseline_head parameters are frozen (requires_grad=False)
[PASS] [attention] pooled GAP matches reference backbone exactly
[PASS] [attention] frozen baseline_logits sub-computation matches reference CornHead exactly
[PASS] [attention] baseline_head parameters are frozen (requires_grad=False)
[PASS] [contrastive] pooled GAP matches reference backbone exactly
[PASS] [contrastive] frozen baseline_logits sub-computation matches reference CornHead exactly
[PASS] [contrastive] baseline_head parameters are frozen (requires_grad=False)
[PASS] 'attention' mode: A_k constant 0.5 at construction
[PASS] [attention] gradient reaches every attention conv'

## 7. Redirect checkpoints/results to Drive

In [9]:
import yaml

OUTPUT_DIR = "/content/drive/MyDrive/thesis/exp10_boundary_specific_representation_outputs"

with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)
cfg["checkpoint_dir"] = f"{OUTPUT_DIR}/checkpoints"
cfg["results_csv"] = f"{OUTPUT_DIR}/results/results.csv"
with open("../configs/base.yaml", "w") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("checkpoint_dir:", cfg["checkpoint_dir"])
print("results_csv:   ", cfg["results_csv"])

checkpoint_dir: /content/drive/MyDrive/thesis/exp10_boundary_specific_representation_outputs/checkpoints
results_csv:    /content/drive/MyDrive/thesis/exp10_boundary_specific_representation_outputs/results/results.csv


## 8. Train Phase A — all 3 modes, all 3 folds

Each fold loads its OWN matching RGB baseline checkpoint, verifies the frozen baseline sub-computation matches the reference `CornHead` exactly before training (stops immediately if not), then trains only the per-boundary residual heads with real early stopping (`patience` from `configs/base.yaml`). MiT-B1 backbone and the baseline's own `CornHead` stay frozen the whole run in every mode (verified via an unchanged-`baseline_logits` probe-batch check after training). Runs sequentially so a later mode's failure doesn't lose an earlier mode's results; each mode's output is appended to the same `results.csv` (keyed by `config_id`+`fold`).

In [10]:
for mode in ["heads", "attention", "contrastive"]:
    print(f"\n===== mode={mode} =====")
    !python train.py --mode {mode} --workers 4


===== mode=heads =====
device=cuda config=mit_b1__corn__boundary_heads folds=3 max_epochs=100 patience=10 contrastive_weight=0.0
[fold0] loading RGB baseline from /content/CanRollasThesisProject/task_3/results/task_3_outputs/checkpoints/mit_b1__corn_fold0.pt
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to c

## 9. Check results

In [11]:
import pandas as pd

df = pd.read_csv(f"{OUTPUT_DIR}/results/results.csv")
display(df)
print(df.groupby("mode")[["accuracy", "macro_f1", "qwk"]].agg(["mean", "std"]))
print()
print("RGB baseline (task_3, Experiment A, not re-run) for reference: "
      "accuracy=0.8168+-0.0150 macro_f1=0.8182+-0.0138 qwk=0.9138+-0.0164")

,config_id,mode,encoder,head,weights,fold,params_m,trainable_params_m,accuracy,macro_f1,mae,qwk,early_stopped_at_epoch,train_time_s
0,mit_b1__corn__boundary_attention,attention,mit_b1,corn,baseline_checkpoint,0,13.352,0.199,0.8011,0.8037,0.2044,0.9093,11,36.0
1,mit_b1__corn__boundary_attention,attention,mit_b1,corn,baseline_checkpoint,1,13.352,0.199,0.8260,0.8288,0.1740,0.9260,11,35.3
2,mit_b1__corn__boundary_attention,attention,mit_b1,corn,baseline_checkpoint,2,13.352,0.199,0.8120,0.8140,0.2016,0.9007,11,36.2
3,mit_b1__corn__boundary_contrastive,contrastive,mit_b1,corn,baseline_checkpoint,0,13.352,0.199,0.8066,0.8085,0.1989,0.9126,12,41.0
4,mit_b1__corn__boundary_contrastive,contrastive,mit_b1,corn,baseline_checkpoint,1,13.352,0.199,0.8370,0.8380,0.1630,0.9317,11,38.5
5,mit_b1__corn__boundary_contrastive,contrastive,mit_b1,corn,baseline_checkpoint,2,13.352,0.199,0.8065,0.8098,0.2071,0.8971,13,44.1
6,mit_b1__corn__boundary_heads,heads,mit_b1,corn,baseline_checkpoint,0,13.350,0.197,0.7956,0.7998,0.2072,0.9092,11,33.0
7,mit_b1__corn__boundary_heads,heads,mit_b1,corn,baseline_checkpoint,1,13.350,0.197,0.8260,0.8281,0.1740,0.9259,12,36.8
8,mit_b1__corn__boundary_heads,heads,mit_b1,corn,baseline_checkpoint,2,13.350,0.197,0.8202,0.8221,0.1962,0.8980,11,34.0


             accuracy            macro_f1                 qwk          
                 mean       std      mean       std      mean       std
mode                                                                   
attention    0.813033  0.012482  0.815500  0.012617  0.912000  0.012864
contrastive  0.816700  0.017580  0.818767  0.016669  0.913800  0.017331
heads        0.813933  0.016140  0.816667  0.014912  0.911033  0.014040

RGB baseline (task_3, Experiment A, not re-run) for reference: accuracy=0.8168+-0.0150 macro_f1=0.8182+-0.0138 qwk=0.9138+-0.0164


## 10. Attention diagnostic (per mode with attention: `attention`, `contrastive`)

Checks the architecture's own stated risk directly: do the three boundaries' learned attention maps actually differ, or did they converge to near-identical spatial focus? Reads checkpoints from `../checkpoints/` (the config still points at Drive from cell 7, so this reads the just-trained Colab checkpoints).

In [12]:
for mode in ["attention", "contrastive"]:
    for fold in range(3):
        !python diagnose_attention.py --mode {mode} --fold {fold} --split test

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
[attention fold0 test] n_images=362
  boun